# EquityAgent: Agent run and evaluation

GitHub repository: https://github.com/nicole-rowley-msaai/MS_AAI_520_NLP_GenAI_Final

This notebook runs the full agent on one ticker and prints every step, then
evaluates it against a single-prompt baseline and with memory on vs off.

## Agent Design and Workflows

The agent is a LangGraph state graph (`src/graph.py`):

```
ticker -> plan -> route -> specialists -> synthesize -> evaluate --pass--> reflect -> memory
                                                            |                 ^
                                                            +--refine--> optimize
```

| Requirement | Where | Printed below |
|---|---|---|
| Plans research steps | `agents/planner.py` | Section 2 |
| Uses tools dynamically | `agents/toolkit.py`, `tools/` | Section 3 |
| Routing | `agents/router.py` | Section 4 |
| Prompt chaining | `chains/news_chain.py` | Section 5 |
| Evaluator-optimizer | `agents/evaluator.py` | Section 7 |
| Self-reflection | `agents/evaluator.py: reflect` | Section 9 |
| Learns across runs | `memory/store.py` | Section 10 |

## 1. Setup

In [ ]:
# --- Colab setup -----------------------------------------------------
import os
import sys

from google.colab import drive, userdata

REPO_URL = (
    f"https://{userdata.get('GITHUB_TOKEN')}@github.com/"
    "nicole-rowley-msaai/MS_AAI_520_NLP_GenAI_Final.git"
)
REPO = "/content/drive/MyDrive/equity-agent"

drive.mount("/content/drive")
if os.path.isdir(os.path.join(REPO, ".git")):
    %cd {REPO}
    !git pull -q
else:
    %cd /content/drive/MyDrive
    !git clone -q {REPO_URL} equity-agent
    %cd {REPO}
sys.path.insert(0, REPO)

SKIP = "google-adk\\|dependency resolver"
!pip install -q -r requirements.txt 2>&1 | grep -v "{SKIP}"

SECRET_KEYS = [
    "OPENAI_API_KEY",
    "ANTHROPIC_API_KEY",
    "FRED_API_KEY",
    "NEWSAPI_KEY",
    "EDGAR_USER_AGENT",
]
loaded = []
for key in SECRET_KEYS:
    try:
        os.environ[key] = userdata.get(key)
        loaded.append(key)
    except Exception:
        print(f"Missing secret: {key}")
print("Loaded:", loaded)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from src import display as show
from src.config import TICKERS
from src.evaluation import (
    baseline_brief, quick_smoke, score_against_findings,
    tool_success_rate, unsupported_figures,
)
from src.graph import run_agent
from src.memory import store
from src.tools.retrieval import index_filings

pd.set_option("display.max_colwidth", 120)

## Agent Functions and Capabilities

### Setup check: every tool returns data, and the filing index is built

In [ ]:
print("Chunks indexed:", index_filings(TICKERS))
pd.DataFrame({t: quick_smoke(t) for t in TICKERS})

## 2. Research plan

The planner reads memory first. On the first run of a ticker there are no
notes; later runs show the lessons it used.

In [ ]:
TICKER = "NVDA"
notes_before = store.load_notes()

report, trace, state = run_agent(TICKER)

print("Sector:", state["sector"])
print("Hypothesis:", state["plan"].hypothesis)
show.plan_table(trace)

In [ ]:
show.memory_read(trace)

## 3. Tool calls (dynamic tool use)

In [ ]:
show.tool_log(trace)

## 4. Routing decisions

In [ ]:
show.routing_table(trace)

## 5. News prompt chain: output of each step

In [ ]:
show.news_chain_steps(trace)

## 6. Specialist findings

In [ ]:
show.findings_table(trace)

## Evaluation and Iteration

## 7. Evaluator-optimizer loop

Scores per iteration, the evaluator's critiques, and the score curve.

In [ ]:
show.evaluation_history(state)

In [ ]:
show.critiques(state)

In [ ]:
show.plot_scores(state)
plt.show()

## 8. Final report

In [ ]:
display(Markdown(report))

## 9. Reflection (self-assessment of output and process)

In [ ]:
show.reflection_view(state)

## 10. Memory: what this run taught the agent

New lessons written to `memory/notes.json`, then a second run on the same
ticker to show the planner reading them.

In [ ]:
show.memory_diff(trace)

In [ ]:
report2, trace2, state2 = run_agent(TICKER)
print("Lessons the planner read on run 2:")
display(show.memory_read(trace2))
print("Run 1 steps:", [s.tool for s in state["plan"].steps])
print("Run 2 steps:", [s.tool for s in state2["plan"].steps])
print("Run 1 final score:", state["history"][-1]["average"],
      "| Run 2 final score:", state2["history"][-1]["average"])

## 11. Evaluation

Three comparisons, all scored with the same rubric and the same findings:

1. **Baseline vs agent:** a single prompt with no tools, no plan, no loop.
2. **Numeric spot-check:** figures in each brief that appear in no finding.
3. **Memory on vs off:** the same ticker with memory disabled.

In [ ]:
rows = []
for t in TICKERS:
    rep, tr, st = run_agent(t)
    base = baseline_brief(t)
    base_score = score_against_findings(base, st["findings"])
    rows.append({
        "ticker": t,
        "baseline_avg": base_score["average"],
        "agent_first_draft": st["history"][0]["average"],
        "agent_final": st["history"][-1]["average"],
        "iterations": len(st["history"]),
        "tool_success": tool_success_rate(tr),
        "unsupported_figs_baseline": len(unsupported_figures(base, st["findings"])),
        "unsupported_figs_agent": len(unsupported_figures(st["draft"], st["findings"])),
    })
results = pd.DataFrame(rows)
results

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
x = range(len(results))
w = 0.27
ax.bar([i - w for i in x], results["baseline_avg"], w, label="baseline", color="#b5b5b5")
ax.bar(list(x), results["agent_first_draft"], w, label="agent, first draft", color="#8fb3e0")
ax.bar([i + w for i in x], results["agent_final"], w, label="agent, final", color="#3b6fb6")
ax.set_xticks(list(x), results["ticker"])
ax.set_ylim(1, 5)
ax.set_ylabel("Rubric average (1-5)")
ax.set_title("Agent beats the single-prompt baseline on every ticker")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8)
plt.show()

In [ ]:
_, tr_off, st_off = run_agent(TICKER, use_memory=False)
_, tr_on, st_on = run_agent(TICKER, use_memory=True)
pd.DataFrame([
    {"memory": "off", "plan_steps": len(st_off["plan"].steps),
     "notes_read": len(tr_off.select("memory", "read")[0].output),
     "final_score": st_off["history"][-1]["average"]},
    {"memory": "on", "plan_steps": len(st_on["plan"].steps),
     "notes_read": len(tr_on.select("memory", "read")[0].output),
     "final_score": st_on["history"][-1]["average"]},
])

In [ ]:
show.timing_table(trace)

## Limitations

- Figures come from free data sources (`yfinance` is unofficial; NewsAPI
  caps at 100 articles) and are cached, so a run reflects the cache date.
- The evaluator is an LLM and its scores vary by about 0.2 between runs.
- The agent has no view of future events, and the brief is analysis only,
  not investment advice.